# Método de Elementos Finitos (FEM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/elementos_finitos/FEM_1D.ipynb)

El Método de Elementos Finitos (FEM) es un método numérico que nos permite computar las soluciones de ecuaciones en dervidas parciales). Para que tenga solución única se necesita el valor de la solución en la frontera (condición de borde). Con esta notebook comenzaremos a desarrollar el método en su versión unidimensional con un ejemplo de utilidad que tiene solución analítica: la ecuación de Poisson.

$$ \left \{ \begin{array}{l} -u^{''}=f \ \ \text{ para } \ \ x\in I=\left(0,1 \right) \\   u(0)=u(1)=0   \end{array} \right . \tag{1}$$

donde $f$ es conocida.

Usaremos el siguiente método para testear el método.

**Ejemplo:** Tomemos $f(x)=1$. En este caso la solución exacta está dada por $u(x)=\frac{x(1-x)}{2}$.

## Formulación variacional

La solución utilizando FEM siempre comienza por reescribir la ecuación (1) en lo que se denominada **forma variacional** o **forma débil**. Multiplicamos la ecuación por una función de prueba  $v$ (*función test*).

$$-\int_{0}^{1}u'' \ \ v \ \ dx = \int_{0}^{1}f \ \ v \ \ dx \tag{2}$$

Notar que la parte izquierda de esta ecuación tiene una segunda derivada, lo que es conveniente bajar el orden de esta derivada (veremos más adelante el porque hacer esto). Para esto se integra por partes

$$-\left[ \left. u'v \ \ \right |_{0}^{1}-\int_{0}^{1}v'u'dx\right] = \int_{0}^{1} fvdx$$

Luego,

$$\int_{0}^{1}v'u'dx + u'(0)v(0) - u'(1)v(1) = \int_{0}^{1} fvdx.$$

A la función $v$ vamos a pedirle que cumpla  $v(0)=0$  y  $v(1)=0$  (si tuvieramos información de $u'$ en los bordes no sería necesario). De esta forma la ecuación anterior nos queda:

$$\int_{0}^{1}v'u' \ \ dx = \int_{0}^{1} fv \ \ dx  \ \ \ \ \forall v \in V_{0}.\tag{3}$$
A $v$ le vamos a pedir que se comporte bien, por eso tendrá que ser:

$$v\in C^{\infty}_c(0,1)$$

Veremos más adelante que no hace falta pedirle tanto.

## Aproximación por elementos finitos

El siguiente paso es aproximar $u$. En este caso usando una función continua y lineal a trozos. Utilizaremos el  $V_h$  (ver la notebook sobre Polinomios a trozos). También le pediremos que se cumpla  $v(0)=0$  y  $v(1)=0$. Entonces resolveremos:

$$\int_{0}^{1}u_{h}^{'}v^{'} \ \ dx = \int_{0}^{1} fv \ \ dx   \ \ \ \ \forall v \in V_{h} \tag{4}$$

A esto se le dice *método de Galerkin*, en honor a quien lo propuso. Observemos que tanto $u_h$  como  $v$  pertenecen al espacio $V_h$ , esto es porque también $u_h$ vale cero en los bordes.
### Sistema de ecuaciones lineales

Lo que vamos a hacer es pasar de una ecuación con integrales (4) a un sistema de ecuaciones lineales cuya solución serán unas constantes  $α_i$, que nos permitirán construir nuestra solución aproximada $u_h$. El sistema de ecuaciones lineales lo expresaremos de la forma:

$$A\alpha = b$$

donde  $A$  es una matriz de  $(n−1)\times (n−1)$ ,  $α$  y  $b$  son vectores de  $(n−1)\times 1$. Tomaremos la función de prueba $v=\phi_i(x)$ con  $i=1,…,n−1$ (funciones bases), dado que $v$ debe ser cero en los bordes (por eso no están  $i=0$  e  $i=n$). Por lo tanto si expresamos la ecuación (4) con esta $v$ tendremos:

$$ \int_{0}^{1}u_{h}'\varphi_{i}'(x) \ \ dx = \int_{0}^{1} f \varphi_{i}(x) \ \ dx   \quad \forall i=1,...,n-1   \tag{5}.$$

Como $u_h$ también pertenece a $V_h$ entonces también la podemos escribir como combinación lineal de las funciones bases ($\phi_j$, le ponemos otro subíndice para distinguirlas):

$$ u_{h} = \sum_{j=1}^{n-1} \alpha_{j} \varphi_{j} \tag{6}.$$
esto lo reemplazo en la ecuación (5) y nos queda

$$ \int_{0}^{1} \left( \sum_{j=1}^{n-1} \alpha_{j} \varphi_{j} \right)' \varphi_{i}' dx = \int_{0}^{1} \sum_{j=1}^{n-1} \alpha_{j} \left( \varphi_{j} \right)' \varphi_{i}' dx = \sum_{j=1}^{n-1} \alpha_{j} \int_{0}^{1} \varphi_{j}' \varphi_{i}' dx = \int_{0}^{1} f \varphi_{i} \ \ dx.$$

Ahora introducimos la notación:

$$A_{ij} = \int_{0}^{1} \varphi_{j}^{'} \varphi_{i}^{'} dx \ \ \text{con} \ \ i,j=1,\dots, n-1 \tag{7}$$

$$b_{i}=\int_{0}^{1} f \varphi_{i} \ \ dx \ \ \text{con} \ \ i=1,\dots, n-1 \tag{8}$$

Luego, nuestro problema se reduce a resolver el siguiente sistema:

$$ \begin{bmatrix} A_{11} & A_{12} & \cdots & A_{1(n-1)} \\ A_{21} & \ddots & \cdots & \vdots \\ \vdots &  &  & \vdots \\  A_{(n-1)1} & \cdots & \cdots & A_{(n-1)(n-1)} \end{bmatrix} \begin{bmatrix} \alpha_{1} \\ \alpha_{2} \\ \vdots \\ \alpha_{n-1} \end{bmatrix}= \begin{bmatrix} b_{1} \\ b_{2} \\ \vdots \\ b_{n-1} \end{bmatrix} \tag{10}$$
A la matriz $A$ se le dice matriz de rigidéz (*stifness matrix*) y a $b$  se le dice vector de cargas (*load vector*)
$

## Algoritmo

El algoritmo deberá realizar las siguientes operaciones:

1. Crear la malla/grilla en $I$, esto determinará el espacio $V_{h}$
2. Computar $A$ y $b$ y armar la matriz $A$ y el vector $b$ como vi anteoriormentease.
3. Resolver el sistema de ecuaci.ones
4. Calcular la solución aproximada: $u_{h}= \sum_{j=1}^{n-1} \alpha_{j} \varphi$

Recordemos que las funciones base (caso lineales) se definen de la siguiente forma:

$$ \varphi_{j}(x_{i})= \left \{ \begin{array}{l} \frac{x-x_{i-1}}{h_{i}} \ \ \text{ si } \ \ x\in I_{i} \\  \frac{x_{i+1}-x}{h_{i+1}}  \ \ \text{ si }  \ \ x\in I_{i+1}\\ 0  \ \ \text{caso contrario} \end{array} \right .$$
$$ \varphi'_{j}(x_{i})= \left \{ \begin{array}{l} \frac{1}{h_{i}} \ \ \text{ si } \ \ x\in I_{i} \\  \frac{-1}{h_{i+1}}  \ \ \text{ si }  \ \ x\in I_{i+1}\\ 0  \ \ \text{caso contrario} \end{array} \right .$$

In [1]:
using LinearAlgebra, Plots

In [ ]:
# cada ejemplo: f, solución exacta u y su derivada du (para medir el error H¹)
ejemplos = [
    (nombre = "f = 1",
     f  = x -> 1.0,
     u  = x -> x*(1-x)/2,
     du = x -> (1-2x)/2),
    (nombre = "u = sin(πx)",
     f  = x -> π^2*sin(π*x),
     u  = x -> sin(π*x),
     du = x -> π*cos(π*x)),
    (nombre = "u = x(1-x)eˣ",
     f  = x -> (3x + x^2)*exp(x),
     u  = x -> x*(1-x)*exp(x),
     du = x -> (1 - x - x^2)*exp(x)),
]

# comparación visual con pocos elementos
ej = ejemplos[1]
z = 0:0.005:1
plt = plot(z, ej.u.(z), lw=2, label="Exacta", title="$(ej.nombre)")
plt

In [ ]:
function LoadVector(N,f)
  h = 1/(N-1)
  x = 0:h:1
  b = zeros(N)
  for k=1:N-1
    s = [0.5*h*f(x[k]),0.5*h*f(x[k+1])]
    b[k:k+1] += s
  end
  return b
end

function Stiffnes(N)
  h = 1/(N-1)
  x = 0:h:1
  A = zeros(N,N)
  for k=1:N-1
    M = [1/h -1/h;-1/h 1/h]
    A[k:k+1,k:k+1] += M
  end
  return A
end

function fem1D_poisson(N,f)
  h = 1/(N-1)
  x = 0:h:1
  A = Stiffnes(N)
  b = LoadVector(N,f)

  #incorporamos el borde
  A[1,:]=[1;zeros(N-1)]
  b[1]=0
  A[N,:]=[zeros(N-1);1]
  b[N]=0

  #resolvemos
  uh = A\b

  return uh
end

In [ ]:
# Bases lineales en el elemento de referencia [0,1]
φ1_P1(ξ) = 1 - ξ
φ2_P1(ξ) = ξ

"""
Agrega a plt la solución P1 evaluada con las bases en cada elemento.
uh tiene N valores nodales (N-1 elementos).
"""
function plot_solucion_P1!(plt, uh; color=:blue, label="FEM")
  Ne = length(uh) - 1
  h = 1/Ne
  ξ = range(0, 1, length=2)
  for k=1:Ne
    c  = uh[k:k+1]
    xx = (k-1)*h .+ h*ξ
    yy = c[1]*φ1_P1.(ξ) .+ c[2]*φ2_P1.(ξ)
    plot!(plt, xx, yy, c=color, lw=2, marker=:o, label = k==1 ? label : "")
  end
  return plt
end

In [ ]:
N = 10
z = 0:0.005:1
plt = plot(z, ej.u.(z), lw=2, c=:black, label="Exacta", title="Exactas vs FEM")
plot_solucion_P1!(plt, fem1D_poisson(N, ej.f))

## Error y orden de convergencia

Para medir qué tan bien aproxima $u_h$ a $u$ usamos dos normas:
$$\|u-u_h\|_{L^2}=\left(\int_0^1 (u-u_h)^2dx\right)^{1/2},\qquad |u-u_h|_{H^1}=\left(\int_0^1 (u'-u_h')^2dx\right)^{1/2}.$$
Para $u$ suficientemente regular y elementos lineales se espera
$$\|u-u_h\|_{L^2}\le C\,h^2,\qquad |u-u_h|_{H^1}\le C\,h .$$
Si $E(h)\approx Ch^p$, entonces dos mallas consecutivas dan el orden estimado
$$p\approx\frac{\log\left(E(h_{i-1})/E(h_i)\right)}{\log\left(h_{i-1}/h_i\right)}.$$
Las integrales de los errores las calculamos elemento a elemento con la cuadratura de Gauss de 3 puntos en $[0,1]$: $\xi=\tfrac12\mp\tfrac12\sqrt{3/5},\,\tfrac12$ con pesos $\tfrac5{18},\tfrac8{18},\tfrac5{18}$.

In [47]:
using Printf

# Cuadratura de Gauss de 3 puntos en [0,1]
ξg = [0.5 - 0.5*sqrt(3/5), 0.5, 0.5 + 0.5*sqrt(3/5)]
wg = [5/18, 8/18, 5/18]

"""
    errores_P1(uh, u, du)

Error en L² y seminorma H¹ de la solución P1 (uh nodal, malla uniforme de N nodos).
"""
function errores_P1(uh, u, du)
  N = length(uh)
  h = 1/(N-1)
  x = range(0, 1, length=N)
  eL2 = 0.0
  eH1 = 0.0
  for k=1:N-1
    xq  = x[k] .+ h*ξg
    uq  = uh[k]*(1 .- ξg) .+ uh[k+1]*ξg      # u_h en los puntos de cuadratura
    duq = (uh[k+1] - uh[k])/h                # u_h' es constante en el elemento
    eL2 += h*sum(wg .* (u.(xq)  .- uq ).^2)
    eH1 += h*sum(wg .* (du.(xq) .- duq).^2)
  end
  return sqrt(eL2), sqrt(eH1)
end

function tabla_orden(hs, E)
  @printf("%10s %12s %7s %12s %7s\n", "h", "‖e‖_L2", "orden", "|e|_H1", "orden")
  for k in eachindex(hs)
    oL2 = k > 1 ? log(E[k-1,1]/E[k,1]) / log(hs[k-1]/hs[k]) : NaN
    oH1 = k > 1 ? log(E[k-1,2]/E[k,2]) / log(hs[k-1]/hs[k]) : NaN
    @printf("%10.3e %12.3e %7.2f %12.3e %7.2f\n", hs[k], E[k,1], oL2, E[k,2], oH1)
  end
end

function plot_orden(hs, E, p; titulo="", sep=3)
  plt = plot(hs, E[:,1], marker=:o, lw=2, xscale=:log10, yscale=:log10,
             label="‖u-uₕ‖_L2", xlabel="h", ylabel="error", title=titulo, legend=:bottomright)
  plot!(plt, hs, E[:,2], marker=:s, lw=2, label="|u-uₕ|_H1")
  # referencias de pendiente, desplazadas por un factor sep
  plot!(plt, hs, (E[end,1]/sep)*(hs/hs[end]).^(p+1), ls=:dash, c=:gray,  label="h^$(p+1)")
  plot!(plt, hs, (E[end,2]*sep)*(hs/hs[end]).^p,     ls=:dot,  c=:black, label="h^$p")
  return plt
end

plot_orden (generic function with 1 method)

In [ ]:
Ns = [2^k + 1 for k in 2:8]          # cantidad de nodos
hs = 1 ./ (Ns .- 1)

for (nombre, f, u, du) in ejemplos
  E = zeros(length(Ns), 2)
  for (i, N) in enumerate(Ns)
    uh = fem1D_poisson(N, f)
    E[i, :] .= errores_P1(uh, u, du)
  end
  println("\nP1 — ", nombre)
  tabla_orden(hs, E)
  display(plot_orden(hs, E, 1; titulo="P1 — $nombre"))
end

## Elementos finitos cuadráticos ($P_2$)

Ahora $V_h$ es el espacio de funciones continuas que son **polinomios de grado $\le 2$ en cada elemento**. Dividimos $I=(0,1)$ en $N_e$ elementos $I_k=[x_k,x_{k+1}]$ de longitud $h=1/N_e$. En cada elemento agregamos el punto medio $x_{k+1/2}=x_k+h/2$, de modo que hay $2N_e+1$ nodos: $0=z_1<z_2<\dots<z_{2N_e+1}=1$. Los nodos con índice impar son los vértices y los de índice par los puntos medios.

**Bases en el elemento de referencia.** Con $x=x_k+h\xi$, $\xi\in[0,1]$, pedimos que cada función valga 1 en su nodo y 0 en los otros dos ($\xi=0,\tfrac12,1$):
$$\hat\varphi_1(\xi)=(1-\xi)(1-2\xi),\qquad \hat\varphi_2(\xi)=4\xi(1-\xi),\qquad \hat\varphi_3(\xi)=\xi(2\xi-1)$$
y sus derivadas respecto de $\xi$ son
$$\hat\varphi_1'=-3+4\xi,\qquad \hat\varphi_2'=4-8\xi,\qquad \hat\varphi_3'=4\xi-1 .$$
Como $\dfrac{d}{dx}=\dfrac1h\dfrac{d}{d\xi}$ y $dx=h\,d\xi$:

**Matriz de rigidez local.** $K^{loc}_{ij}=\displaystyle\int_{I_k}\varphi_i'\varphi_j'\,dx=\frac1h\int_0^1\hat\varphi_i'\hat\varphi_j'\,d\xi$. Por ejemplo,
$$K^{loc}_{11}=\frac1h\int_0^1(-3+4\xi)^2d\xi=\frac{7}{3h},\qquad K^{loc}_{12}=\frac1h\int_0^1(-3+4\xi)(4-8\xi)d\xi=-\frac{8}{3h},$$
y completando las demás entradas queda
$$K^{loc}=\frac{1}{3h}\begin{bmatrix}7&-8&1\\-8&16&-8\\1&-8&7\end{bmatrix}.$$

**Vector de carga local.** $b^{loc}_i=\displaystyle\int_{I_k}f\varphi_i\,dx$. Aproximamos con la regla de Simpson (análoga al trapecio que usamos en $P_1$); como $\hat\varphi_i$ vale $1$ en su nodo y $0$ en los otros,
$$b^{loc}\approx\frac h6\begin{bmatrix}f(x_k)\\4f(x_{k+1/2})\\f(x_{k+1})\end{bmatrix}.$$

**Ensamblado.** El elemento $k$ aporta a los nodos $2k-1,\,2k,\,2k+1$: la matriz local se suma sobre el bloque $A[2k-1:2k+1,\,2k-1:2k+1]$ y el vector local sobre $b[2k-1:2k+1]$. Los bloques de elementos vecinos comparten el nodo $2k+1$. El resto es igual que en $P_1$: se imponen las condiciones de borde y se resuelve $A\alpha=b$.

In [ ]:
# Bases cuadráticas en el elemento de referencia [0,1]
φ1(ξ) = (1 - ξ)*(1 - 2ξ)
φ2(ξ) = 4ξ*(1 - ξ)
φ3(ξ) = ξ*(2ξ - 1)

# Derivadas respecto de ξ
dφ1(ξ) = -3 + 4ξ
dφ2(ξ) = 4 - 8ξ
dφ3(ξ) = 4ξ - 1

ξ = 0:0.01:1
plot(ξ, φ1.(ξ), lw=2, label="φ₁")
plot!(ξ, φ2.(ξ), lw=2, label="φ₂")
plot!(ξ, φ3.(ξ), lw=2, label="φ₃", xlabel="ξ", title="Bases P2 en el elemento de referencia")

In [49]:
function LoadVector_P2(Ne, f)
  h = 1/Ne
  x = range(0, 1, length=2Ne+1)
  b = zeros(2Ne+1)
  for k=1:Ne
    i = 2k-1
    s = (h/6)*[f(x[i]), 4*f(x[i+1]), f(x[i+2])]
    b[i:i+2] += s
  end
  return b
end

function Stiffnes_P2(Ne)
  h = 1/Ne
  A = zeros(2Ne+1, 2Ne+1)
  M = [7 -8 1; -8 16 -8; 1 -8 7]/(3h)
  for k=1:Ne
    i = 2k-1
    A[i:i+2, i:i+2] += M
  end
  return A
end

function fem1D_poisson_P2(Ne, f)
  A = Stiffnes_P2(Ne)
  b = LoadVector_P2(Ne, f)
  N = 2Ne+1

  #incorporamos el borde
  A[1,:]=[1;zeros(N-1)]
  b[1]=0
  A[N,:]=[zeros(N-1);1]
  b[N]=0

  #resolvemos
  uh = A\b
  return uh
end

fem1D_poisson_P2 (generic function with 1 method)

In [ ]:
"""
Agrega a plt la solución P2 evaluada con las bases en cada elemento
(no alcanza con unir los nodos con rectas: u_h es cuadrática a trozos).
"""
function plot_solucion_P2!(plt, uh, Ne; color=:red, label="Solución P2")
  h = 1/Ne
  ξ = range(0, 1, length=30)
  for k=1:Ne
    c  = uh[2k-1:2k+1]
    xx = (k-1)*h .+ h*ξ
    yy = c[1]*φ1.(ξ) .+ c[2]*φ2.(ξ) .+ c[3]*φ3.(ξ)
    plot!(plt, xx, yy, c=color, lw=2, label = k==1 ? label : "")
  end
  return plt
end
ej = ejemplos[3]
# Comparación con 4 elementos: P1 (5 nodos) vs P2 (9 nodos)
z = 0:0.005:1
plt = plot(z, ej.u.(z), lw=2, c=:black, label="Exacta", title="4 elementos: P1 vs P2")
plot!(plt, range(0,1,length=5), fem1D_poisson(5, ej.f), marker=:o, label="P1")
plot_solucion_P2!(plt, fem1D_poisson_P2(4, ej.f), 4)
plt

In [ ]:
"""
    errores_P2(uh, u, du)

Error en L² y seminorma H¹ de la solución P2 (uh con 2Ne+1 valores nodales).
"""
function errores_P2(uh, u, du)
  Ne = (length(uh)-1) ÷ 2
  h = 1/Ne
  eL2 = 0.0
  eH1 = 0.0
  for k=1:Ne
    c   = uh[2k-1:2k+1]
    xq  = (k-1)*h .+ h*ξg
    uq  = c[1]*φ1.(ξg) .+ c[2]*φ2.(ξg) .+ c[3]*φ3.(ξg)
    duq = (c[1]*dφ1.(ξg) .+ c[2]*dφ2.(ξg) .+ c[3]*dφ3.(ξg))/h
    eL2 += h*sum(wg .* (u.(xq)  .- uq ).^2)
    eH1 += h*sum(wg .* (du.(xq) .- duq).^2)
  end
  return sqrt(eL2), sqrt(eH1)
end

Nes = [2^k for k in 2:7]             # cantidad de elementos
hs2 = 1 ./ Nes

E2 = zeros(length(Nes), 2)
for (i, Ne) in enumerate(Nes)
  uh = fem1D_poisson_P2(Ne, ej.f)
  E2[i, :] .= errores_P2(uh, ej.u, ej.du)
end

println("\nP2 — u")
tabla_orden(hs2, E2)
plot_orden(hs2, E2, 2; titulo="P2 — u")

In [ ]:
uh = fem1D_poisson_P2(8, ejemplo[1].f)
x  = range(0, 1, length=17)
println("máx |u - uₕ| en los nodos (f = 1, P2): ", maximum(abs.(u1.(x) .- uh)))   # ~1e-16

Con $f=1$ la solución exacta $u=\tfrac{x(1-x)}{2}$ es un polinomio cuadrático, entonces **pertenece a $V_h$** para $P_2$ y el método la reproduce exactamente (el error es solo redondeo, $\sim10^{-16}$). Con $P_1$ en cambio el error en $L^2$ sigue siendo $O(h^2)$.
Con $u=\sin(\pi x)$ se ve la mejora al pasar de $P_1$ a $P_2$: los órdenes pasan de $(2,1)$ a $(3,2)$ en $(L^2,H^1)$.